# 03 — Preprocessing & Feature Engineering

Prepare the dataset for clustering: filter, clean text, and generate three embedding types (TF-IDF, MiniLM, KaLM).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import spacy

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

raw_path = project_root / "data" / "raw" / "arxiv_papers.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_path)
print(f"Loaded: {len(df):,} papers")

Loaded: 226,879 papers


## Filter to Target Categories

In [2]:
target_cats = {
    "cs.AI",
    "cs.LG",
    "cs.CL",
    "cs.CV",
    "cs.RO",
    "cs.NE",
    "cs.MA",
    "cs.IR",
}

df = df[df["primary_category"].isin(target_cats)].copy()
print(f"After category filter: {len(df):,} papers")

df = df[df["abstract"].str.split().str.len() >= 30].copy()
print(f"After dropping short abstracts: {len(df):,} papers")

df = df.reset_index(drop=True)

After category filter: 181,335 papers
After dropping short abstracts: 181,294 papers


## Clean Text

Two versions:
- `abstract_clean`: whitespace normalized (for neural models)
- `abstract_lemma`: lowercased, stopwords removed, lemmatized (for TF-IDF)

In [3]:
df["abstract_clean"] = df["abstract"].str.replace(r"\s+", " ", regex=True).str.strip()

print(f"Sample:\n{df['abstract_clean'].iloc[0][:200]}...")

Sample:
We introduce MediX-R1, an open-ended Reinforcement Learning (RL) framework for medical multimodal large language models (MLLMs) that enables clinically grounded, free-form answers beyond multiple-choi...


In [4]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])


def lemmatize(text):
    """Lowercase, remove stopwords, lemmatize."""
    doc = nlp(text.lower())
    return " ".join(
        t.lemma_ for t in doc if not t.is_stop and t.is_alpha and len(t) > 2
    )


# Process in batches using spaCy's pipe for speed
print("Lemmatizing (this takes a few minutes)...")
df["abstract_lemma"] = [
    " ".join(t.lemma_ for t in doc if not t.is_stop and t.is_alpha and len(t) > 2)
    for doc in nlp.pipe(df["abstract_clean"], batch_size=1000)
]

print(f"Done. Sample:\n{df['abstract_lemma'].iloc[0][:200]}...")

Lemmatizing (this takes a few minutes)...
Done. Sample:
introduce MediX open ended Reinforcement Learning framework medical multimodal large language model mllm enable clinically ground free form answer multiple choice format MediX fine tune baseline visio...


In [5]:
df["abstract_lemma"] = df["abstract_lemma"].str.lower()
print(f"Sample:\n{df['abstract_lemma'].iloc[0][:200]}...")

Sample:
introduce medix open ended reinforcement learning framework medical multimodal large language model mllm enable clinically ground free form answer multiple choice format medix fine tune baseline visio...


In [7]:
df.to_csv(processed_dir / "arxiv_clean.csv", index=False)
print(f"Saved {len(df):,} papers to {processed_dir / 'arxiv_clean.csv'}")

Saved 181,294 papers to /home/dino/dev/learning/iu-bsc/unsupervised-learning/arxiv-ai-trends/data/processed/arxiv_clean.csv


## TF-IDF Vectorization

In [8]:
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=50000, min_df=5, max_df=0.95)
tfidf_matrix = tfidf.fit_transform(df["abstract_lemma"])

sparse.save_npz(processed_dir / "tfidf_vectors.npz", tfidf_matrix)

print(f"TF-IDF shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")

TF-IDF shape: (181294, 27401)
Vocabulary size: 27,401


## MiniLM Embeddings (BERT-family baseline)

In [9]:
from sentence_transformers import SentenceTransformer

model_mini = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

print(f"Embedding {len(df):,} abstracts with MiniLM...")
minilm_embeddings = model_mini.encode(
    df["abstract_clean"].tolist(),
    batch_size=256,
    show_progress_bar=True,
)

np.save(processed_dir / "minilm_embeddings.npy", minilm_embeddings)
print(f"MiniLM shape: {minilm_embeddings.shape}")

/home/dino/miniforge3/envs/arxiv-trends/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████| 103/103 [00:00<00:00, 1300.05it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 181,294 abstracts with MiniLM...


Batches: 100%|█████████████████████████████████████████████████████████████████████████████| 709/709 [04:49<00:00,  2.45it/s]


MiniLM shape: (181294, 384)


## KaLM Embeddings (state-of-the-art)

In [10]:
model_kalm = SentenceTransformer(
    "HIT-TMG/KaLM-embedding-multilingual-mini-instruct-v1.5",
    device="cuda",
)

print(f"Embedding {len(df):,} abstracts with KaLM...")
kalm_embeddings = model_kalm.encode(
    df["abstract_clean"].tolist(),
    batch_size=64,
    show_progress_bar=True,
)

np.save(processed_dir / "kalm_embeddings.npy", kalm_embeddings)
print(f"KaLM shape: {kalm_embeddings.shape}")

Loading weights: 100%|██████████████████████████████████| 290/290 [00:00<00:00, 1216.79it/s, Materializing param=norm.weight]


Embedding 181,294 abstracts with KaLM...


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 2833/2833 [2:15:02<00:00,  2.86s/it]


KaLM shape: (181294, 896)


## Summary

Preprocessed **181,294** papers (filtered from 226,879 to 8 target primary categories, dropped 0 short abstracts).

**Text cleaning:**
- `abstract_clean`: whitespace normalized (for neural models)
- `abstract_lemma`: lowercased, stopwords removed, lemmatized via spaCy (for TF-IDF)

**Embeddings generated:**
- TF-IDF: 181,294 x 27,401 (sparse)
- MiniLM (all-MiniLM-L6-v2): 181,294 x 384
- KaLM (KaLM-multilingual-mini-instruct-v1.5): 181,294 x 896

All saved to `data/processed/`.

**Next:** Dimensionality reduction in `04_dimensionality_reduction.ipynb`